
# 練習問題: GPUで行列積の性能測定 (CPU との比較)

## 目標

意味のある計算 (行列積 `C = A * B`) を GPU で実行し, **GFLOPS で性能を測定**して CPU (ホスト) 実行と比較する. 同じソース・同じディレクティブが, 環境変数だけで CPU でも GPU でも走ることを体感する.

## 課題

`gpu_matmul.cpp` (または `gpu_matmul.f90`) は行列積を計算する完成済みプログラムである (オフロード指示は既に書かれている). 浮動小数点演算数は `2 n³` で, `omp_get_wtime` で経過時間を測り GFLOPS を表示する.

このプログラムを **GPU 実行**と **CPU (ホスト) 実行**の両方で, いくつかの `n` について動かし, GFLOPS を比較せよ.

- GPU 実行: `OMP_TARGET_OFFLOAD=MANDATORY` (オフロードを強制. 未指定でも `-mp=gpu` でビルドすれば GPU で走る).
- CPU 実行: `OMP_TARGET_OFFLOAD=DISABLED` とし, `OMP_NUM_THREADS` でスレッド数を指定 (1 または 32 の倍数).

## コンパイルと実行

```
# C++
nvc++ -fast -mp=gpu gpu_matmul.cpp -o gpu_matmul.exe

# Fortran
nvfortran -fast -mp=gpu gpu_matmul.f90 -o gpu_matmul.exe
```

GPU は計算ノードにのみ搭載されているので `%%bash_submit` でジョブとして投入する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

echo "=== GPU ==="
for n in 512 1024 2048; do
  OMP_TARGET_OFFLOAD=MANDATORY ./gpu_matmul.exe $n
done

echo "=== CPU (32 threads) ==="
for n in 512 1024 2048; do
  OMP_TARGET_OFFLOAD=DISABLED OMP_NUM_THREADS=32 ./gpu_matmul.exe $n
done
```

## 期待される結果

各行に `n`, 経過時間, GFLOPS, 検算 (`OK`) が表示される.

```
=== GPU ===
n = 512, elapsed = ..., ... GFLOPS, OK
...
```

考えどころ:

- `n` が小さいうちは, GPU はデータ転送・カーネル起動のオーバーヘッドが効いて CPU に対する優位が小さい. `n` を大きくすると GPU の GFLOPS が伸びてくる.
- `05_speedup/01_matmul` の CPU 行列積 (スレッド数を増やしての台数効果) と, ここでの GPU の GFLOPS を比べてみよ. GPU の方がどの程度速いか, どの `n` から有利になるかを考察せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ gpu_matmul.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 1024);
  double * A = (double *)malloc(sizeof(double) * n * n);
  double * B = (double *)malloc(sizeof(double) * n * n);
  double * C = (double *)malloc(sizeof(double) * n * n);
  for (long i = 0; i < n * n; i++) { A[i] = 1.0; B[i] = 1.0; C[i] = 0.0; }

  double t0 = omp_get_wtime();
  /* 行列積 C = A * B (完成済み).
     OMP_TARGET_OFFLOAD=DISABLED ならホスト(CPU)で, MANDATORY ならGPUで実行される. */
#pragma omp target teams distribute parallel for map(to: A[0:n*n], B[0:n*n]) map(from: C[0:n*n])
  for (long i = 0; i < n; i++) {
    for (long j = 0; j < n; j++) {
      double s = 0.0;
      for (long k = 0; k < n; k++) s += A[i * n + k] * B[k * n + j];
      C[i * n + j] = s;
    }
  }
  double t1 = omp_get_wtime();
  double dt = t1 - t0;

  /* 検算: A,B が全て 1 なので C[i][j] = n になるはず */
  long err = 0;
  for (long i = 0; i < n * n; i++) if (C[i] != (double)n) err++;

  double flops = 2.0 * (double)n * (double)n * (double)n;
  printf("n = %ld, elapsed = %.3f sec, %.3f GFLOPS, %s\n",
         n, dt, flops / dt * 1e-9, (err == 0 ? "OK" : "NG"));
  free(A); free(B); free(C);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu gpu_matmul.cpp -o gpu_matmul_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_matmul_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ gpu_matmul.f90
program gpu_matmul
  use omp_lib
  character(len=32) :: arg
  integer(8) :: n, i, j, k, err
  real(8), allocatable :: A(:,:), B(:,:), C(:,:)
  real(8) :: s, t0, t1, dt, flops
  n = 1024
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(A(n,n), B(n,n), C(n,n))
  A = 1.0d0; B = 1.0d0; C = 0.0d0

  t0 = omp_get_wtime()
  ! 行列積 C = A * B (完成済み).
  ! OMP_TARGET_OFFLOAD=DISABLED ならホスト(CPU)で, MANDATORY ならGPUで実行される.
  !$omp target teams distribute parallel do map(to: A, B) map(from: C) private(j, k, s)
  do i = 1, n
     do j = 1, n
        s = 0.0d0
        do k = 1, n
           s = s + A(i,k) * B(k,j)
        end do
        C(i,j) = s
     end do
  end do
  !$omp end target teams distribute parallel do
  t1 = omp_get_wtime()
  dt = t1 - t0

  ! 検算: A,B が全て 1 なので C(i,j) = n になるはず
  err = 0
  do j = 1, n
     do i = 1, n
        if (C(i,j) /= dble(n)) err = err + 1
     end do
  end do

  flops = 2.0d0 * dble(n) * dble(n) * dble(n)
  if (err == 0) then
     print "(a,i0,a,f0.3,a,f0.3,a)", "n = ", n, ", elapsed = ", dt, " sec, ", flops / dt * 1.0d-9, " GFLOPS, OK"
  else
     print "(a,i0,a,f0.3,a,f0.3,a)", "n = ", n, ", elapsed = ", dt, " sec, ", flops / dt * 1.0d-9, " GFLOPS, NG"
  end if
end program gpu_matmul

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu gpu_matmul.f90 -o gpu_matmul_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_matmul_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:gpu_matmul.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gpu_matmul.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gpu_matmul.cpp}